In [14]:
import sqlite3
import pandas as pd
import numpy as np

In [15]:
conn= sqlite3.connect('Saudi_Power.dp')
df_electricity= pd.read_excel('Saudi_Power_Consumption_Factors.xlsx', sheet_name= 'Electricity')
df_temperature= pd.read_excel('Saudi_Power_Consumption_Factors.xlsx', sheet_name= 'Temperature')
df_population= pd.read_excel('Saudi_Power_Consumption_Factors.xlsx', sheet_name= 'Population')
df_electricity.head()
df_electricity.info()
df_electricity.duplicated().sum()
df_temperature.head()
df_temperature.info()
df_temperature.duplicated().sum()
df_population.head()
df_population.info()
df_population.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column                                   Non-Null Count  Dtype 
---  ------                                   --------------  ----- 
 0   Years                                    20 non-null     int64 
 1   Region                                   20 non-null     object
 2   Electricity consumption in MWh\Customer  20 non-null     int64 
dtypes: int64(2), object(1)
memory usage: 612.0+ bytes
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 4 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Years                       20 non-null     int64  
 1   Region                      20 non-null     object 
 2   Avgerage Temperature in C°  20 non-null     float64
 3   Maximum Temperature in C°   20 non-null     float64
dtypes: float64(2), int64(1), object(1)
memory usage: 

np.int64(0)

In [16]:
df_electricity.to_sql('Electricity', conn, if_exists= 'replace', index= False)
df_temperature.to_sql('Temperature', conn, if_exists= 'replace', index= False)
df_population.to_sql('Population', conn, if_exists= 'replace', index= False)

20

In [17]:
query= r'''
SELECT
   e.Region,
   e.Years,
   e.[Electricity Consumption in MWh\Customer] AS MWh_Per_Customer,
   t.[Avgerage Temperature in C°] As Avg_Temp,
   t.[Maximum Temperature in C°] As Max_Temp,
   p.Population

FROM Electricity e
JOIN Temperature t
     ON e.Region = t.Region AND e.Years = t.Years
JOIN Population p
     ON e.Region = p.Region AND e.Years = p.Years
ORDER BY e.Region, e.Years;
'''
df_master= pd.read_sql_query(query, conn)
df_master

,Region,Years,MWh_Per_Customer,Avg_Temp,Max_Temp,Population
0,Central,2020,27,26.7,46.3,10452563
1,Central,2021,28,27.5,45.9,10179832
2,Central,2022,28,27.2,46.0,10674333
3,Central,2023,29,27.1,46.0,11218394
4,Central,2024,28,27.4,47.6,11793138
5,Eastern,2020,46,26.9,46.8,4989327
6,Eastern,2021,45,27.6,47.7,4879962
7,Eastern,2022,45,27.2,47.0,5125254
8,Eastern,2023,47,27.4,48.3,5393097
9,Eastern,2024,48,27.5,48.2,5673507


In [19]:
df_master.isnull().sum()
df_master.duplicated().sum()
df_master.dtypes
df_master

,Region,Years,MWh_Per_Customer,Avg_Temp,Max_Temp,Population
0,Central,2020,27,26.7,46.3,10452563
1,Central,2021,28,27.5,45.9,10179832
2,Central,2022,28,27.2,46.0,10674333
3,Central,2023,29,27.1,46.0,11218394
4,Central,2024,28,27.4,47.6,11793138
5,Eastern,2020,46,26.9,46.8,4989327
6,Eastern,2021,45,27.6,47.7,4879962
7,Eastern,2022,45,27.2,47.0,5125254
8,Eastern,2023,47,27.4,48.3,5393097
9,Eastern,2024,48,27.5,48.2,5673507


In [10]:
from sklearn.linear_model import LinearRegression
future_years = [2025, 2026, 2027, 2028]
regions = df_master['Region'].unique()
results = []

In [12]:
for region in regions:

    df_region = df_master[df_master['Region'] == region].sort_values('Years')
    X = df_region[['Population','Avg_Temp']]
    y = df_region['MWh_Per_Customer']
    ml_model = LinearRegression()
    ml_model.fit(X, y)

    last_row = df_region.iloc[-1]
    last_pop = last_row['Population']
    last_mwh = last_row['MWh_Per_Customer']
    avg_temp = df_region['Avg_Temp'].mean()
    max_temp = df_region['Max_Temp'].mean()

    pop_growth_rate = 0.025

    for idx, year in enumerate(future_years, 1):
        future_pop = int(last_pop * (1 + pop_growth_rate) ** idx)
        future_temp = avg_temp + (0.05*idx)

        future_data = pd.DataFrame({
            'Population': [future_pop],
            'Avg_Temp': [future_temp]
            })
        pred_mwh = ml_model.predict(future_data)[0]
        if pred_mwh < last_mwh:
            pred_mwh = last_mwh * (1 + 0.015 * idx)

        results.append({
            'Region': region,
            'Years': year,
            'Avg_Temp': round(future_temp, 2),
            'Max_Temp': round(max_temp, 2),
            'Population': future_pop,
            'MWh_Per_Customer_Forecast': round(pred_mwh, 2)
        })

df_forecast = pd.DataFrame(results)
display(df_forecast)

,Region,Years,Avg_Temp,Max_Temp,Population,MWh_Per_Customer_Forecast
0,Central,2025,27.23,46.36,12087966,28.51
1,Central,2026,27.28,46.36,12390165,28.67
2,Central,2027,27.33,46.36,12699919,28.83
3,Central,2028,27.38,46.36,13017417,29.00
4,Eastern,2025,27.37,47.60,5815344,48.47
5,Eastern,2026,27.42,47.60,5960728,49.00
6,Eastern,2027,27.47,47.60,6109746,49.54
7,Eastern,2028,27.52,47.60,6262490,50.10
8,Southern,2025,18.27,31.54,4886831,22.33
9,Southern,2026,18.32,31.54,5009001,22.22


In [13]:
with pd.ExcelWriter('Saudi_Power_SQL_Forecast_2028.xlsx') as writer:
  df_master.to_excel(writer, sheet_name='Historical_SQL_Data', index=False)
  df_forecast.to_excel(writer, sheet_name='Forecast_2025_2028', index=False)

  print('Forecast Generated Successfully!')

Forecast Generated Successfully!
